# Stg Layer — place_rating_reads (every Google rating read, append-only)

Added 2026-11-01. `stg_places` (full rebuild) and `stg_restaurant_place_details` (MERGE on `place_id`) keep only the **latest** Google rating of a place, so a change (e.g. 4.9★ on day 1, 4.6★ a month later) was lost. This table keeps **every read**: one row per `place_id` × `read_at` × `source`, never updated or deleted.

**Sources:**
- `place_details` = `stg_restaurant_place_details` (rating / `rating_count` of the latest Place Details read, `fetched_at`). This task runs right after it every day, so each read is caught before the next one overwrites it.
- `text_search` = `bronze.places_raw` (manual Places searches return the rating, Enterprise SKU; `auto_search` uses Text Search Pro and returns none, so those rows are skipped). Bronze is append-only, so this also fills the reads before 2026-11-01; Bronze keeps 180 days, this table keeps them forever.

**Load:** insert-only `MERGE` on (`place_id`, `read_at`, `source`): a rerun adds nothing twice. Rows without a rating are skipped.

**How often a place is read** (Place Details, `MONTHLY_CALL_LIMIT` 900): restaurants with a YouTube review every 30 days, the top 20 by `legit_score` every 7 days, Google-only places every 90 days. `gold/fact_restaurant_rating_daily_snapshot` turns these reads into one row per restaurant × day.

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

PLACES_RAW = f"{BRONZE}.places_raw"
DETAILS_TABLE = f"{SILVER}.stg_restaurant_place_details"
TARGET = f"{SILVER}.stg_place_rating_reads"
print(f"Sources: {DETAILS_TABLE}, {PLACES_RAW} · target: {TARGET}")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET} (
        place_id      STRING,
        read_at       TIMESTAMP,
        source        STRING,
        rating        DOUBLE,
        rating_count  INT,
        etl_date      TIMESTAMP
    )
    USING DELTA
    COMMENT 'Every Google rating read of a place (append-only). Grain: place_id x read_at x source (place_details / text_search).'
""")

parts = []
if spark.catalog.tableExists(DETAILS_TABLE):
    parts.append(f"""
        SELECT place_id, fetched_at AS read_at, 'place_details' AS source,
               CAST(rating AS DOUBLE) AS rating, CAST(rating_count AS INT) AS rating_count
        FROM {DETAILS_TABLE}
        WHERE fetch_status = 'ok' AND rating IS NOT NULL AND fetched_at IS NOT NULL""")
if spark.catalog.tableExists(PLACES_RAW):
    parts.append(f"""
        SELECT place_id, COALESCE(updated_at, created_at) AS read_at, 'text_search' AS source,
               CAST(rating AS DOUBLE) AS rating, CAST(user_rating_count AS INT) AS rating_count
        FROM {PLACES_RAW}
        WHERE rating IS NOT NULL AND COALESCE(updated_at, created_at) IS NOT NULL""")

before = spark.table(TARGET).count()
if parts:
    # Bronze can hold the same Lakebase row twice (re-ingested): one row per grain, latest values
    spark.sql(f"""
        SELECT place_id, read_at, source, max_by(rating, rating_count) AS rating, max(rating_count) AS rating_count
        FROM ({' UNION ALL '.join(parts)})
        WHERE place_id IS NOT NULL
        GROUP BY place_id, read_at, source
    """).createOrReplaceTempView("rating_reads_src")
    spark.sql(f"""
        MERGE INTO {TARGET} AS t
        USING rating_reads_src AS s
          ON t.place_id = s.place_id AND t.read_at = s.read_at AND t.source = s.source
        WHEN NOT MATCHED THEN INSERT (place_id, read_at, source, rating, rating_count, etl_date)
        VALUES (s.place_id, s.read_at, s.source, s.rating, s.rating_count, current_timestamp())
    """)
else:
    print("No source table yet: nothing to read")
after = spark.table(TARGET).count()
print(f"✅ {TARGET}: +{after - before} new reads ({after} in total)")

In [ ]:
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT place_id, read_at, source) AS grain, COUNT(DISTINCT place_id) AS places,
           COUNT(DISTINCT CASE WHEN n_reads > 1 THEN place_id END) AS places_read_twice
    FROM (SELECT *, COUNT(*) OVER (PARTITION BY place_id) AS n_reads FROM {TARGET})
""").collect()[0]
print(f"📊 {s['places']} places · {s['places_read_twice']} read more than once · grain check: " +
      ("✅" if s["n"] == s["grain"] else "❌ duplicates"))
if s["n"] != s["grain"]:
    raise AssertionError(f"{TARGET}: {s['n'] - s['grain']} duplicate (place_id, read_at, source) rows")
display(spark.sql(f"SELECT source, to_date(read_at) AS read_date, COUNT(*) AS reads FROM {TARGET} GROUP BY 1, 2 ORDER BY 2 DESC, 1 LIMIT 30"))
print("\n--- stg_place_rating_reads complete ---")